In [ ]:
# Environment setup — run this cell first before any other imports.
# Required because flash-attn needs a newer libstdc++ than the system provides.
import ctypes, os, sys

_LIBSTDCXX = "/opt/py/mamba/pkgs/libstdcxx-15.2.0-h934c35e_15/lib/libstdc++.so.6"
_TX1_ROOT   = "/home/MACHEN/tahoe-x1-work/tahoe-x1-install"

ctypes.CDLL(_LIBSTDCXX)                          # load newer libstdc++ before flash-attn
if _TX1_ROOT not in sys.path:
    sys.path.insert(0, _TX1_ROOT)                # make `scripts.*` importable

print("Environment ready.")


# Perturbation response prediction with Tx1 + STATE

Example of a perturbation prediction pipeline using Tx1 embeddings as the representation space
and Arc's [State Transition](https://github.com/ArcInstitute/state) (ST) model.

The overall workflow is to first generate Tx1 embeddings for your training data (including perturbed
and control cells), use those embeddings to train an ST model, and use the trained ST model
for downstream inference and evaluation.

This tutorial assumes you have the `arc-state` package installed.

### 1. Prepare the input AnnData

This should be an AnnData with both control and perturbed cells. Below are examples of columns to be included.

| field | meaning | example |
| --- | --- | --- |
| `pert_col` | `.obs` perturbation label | `"drugname_drugconc"` (drug) or `"target_gene"` (CRISPR) |
| `control_pert` | value of `pert_col` marking controls | `"DMSO_TF"` or `"non-targeting guide"` |
| `cell_type_key` | `.obs` cell line or cell type | `"cell_line_id"` or `"cell_type"` |
| `gene_id_key` | `.var` Ensembl IDs (for Tx1) | `"ensembl_id"` |

`adata.X` should be counts since Tx1 does its own binning.

In comparison to `state tx preprocess_train`, which builds `.obsm["X_hvg"]`, we are
using Tx1 features instead, so we skip `preprocess_train` and write Tx1
embeddings into `.obsm` ourselves in step 2. Keep gene expression in `adata.X` or another layer if
you want to predict in the gene space.

In [ ]:
import scanpy as sc

ADATA_PATH = "/path/to/your_data.h5ad"
PERT_COL = "drugname_drugconc"
CONTROL_PERT = "DMSO_TF"
CELL_TYPE_KEY = "cell_line_id"
GENE_ID_KEY = "ensembl_id"

adata = sc.read_h5ad(ADATA_PATH)
print(adata)
assert PERT_COL in adata.obs and CONTROL_PERT in set(adata.obs[PERT_COL])
assert GENE_ID_KEY in adata.var
print("perturbations:", adata.obs[PERT_COL].nunique(), "| cell types:", adata.obs[CELL_TYPE_KEY].nunique())

### 2. Generate Tahoe-x1 embeddings

We copy the embedding into a stable key `EMBED_KEY` that every `state` command will reference.

In [ ]:
import os
from omegaconf import OmegaConf as om
from scripts.inference.predict_embeddings import predict_embeddings

MODEL_SIZE = "70m"
MODEL_NAME = f"Tx1-{MODEL_SIZE}"
EMBED_KEY = f"tx1-{MODEL_SIZE}"
DATASET_DIR = "/path/to/output/dir/"

cfg = om.create({
    "model_name": MODEL_NAME,
    "paths": {"hf_repo_id": "tahoebio/Tahoe-x1",
              "hf_model_size": MODEL_SIZE,
              "adata_input": ADATA_PATH},
    "data": {"cell_type_key": CELL_TYPE_KEY, "gene_id_key": GENE_ID_KEY},
    "predict": {"seq_len_dataset": 2048, "return_gene_embeddings": False},
})

adata = predict_embeddings(cfg)
adata.obsm[EMBED_KEY] = adata.obsm[MODEL_NAME]
print(f"stored embeddings at adata.obsm[{EMBED_KEY}]")

os.makedirs(DATASET_DIR, exist_ok=True)
adata.write_h5ad(os.path.join(DATASET_DIR, "perturb_with_tx_emb.h5ad"))

### 3. Configure data split TOML for STATE

This follows the standard STATE format. `[zeroshot]` holds out whole cell types and `[fewshot]` holds out specific
perturbations within a cell type. (See `examples/zeroshot.toml`, `examples/fewshot.toml`, and the
`state_preprint_toml_files/` in the STATE repo.)

In [ ]:
[datasets]
tahoe_pert = "/path/to/output/dir/"

[training]
tahoe_pert = "train"

[zeroshot]
"tahoe_pert.<HELD_OUT_CELL_TYPE>" = "test"

# or, instead of zeroshot:
# [fewshot]
# [fewshot."tahoe_pert.<CELL_TYPE>"]
# val = ["<PERT_A>"]
# test = ["<PERT_B>"]

### 4. Train the ST model on Tx1 embeddings

We point `data.kwargs.embed_key` to the embedding key from step 2.

In [ ]:
state tx train \
  data.kwargs.toml_config_path="config.toml" \      # point to file from step 3
  data.kwargs.embed_key="tx1-70m" \                 # same as EMBED_KEY
  data.kwargs.pert_col="drugname_drugconc" \        # same as PERT_COL
  data.kwargs.cell_type_key="cell_line_id" \        # same as CELL_TYPE_KEY
  data.kwargs.control_pert="DMSO_TF" \              # same as CONTROL_PERT
  data.kwargs.output_space="gene" \
  model=state \
  training.max_steps=300000 \
  training.batch_size=64 \
  training.lr=1e-4 \
  output_dir="/path/to/state/output/" \
  name="st-tx1-70m"

You can then evaluate the model using `state tx predict`, which uses Arc's
[`cell-eval`](https://github.com/ArcInstitute/cell-eval) metrics.

In [ ]:
state tx predict --output-dir "/path/to/state/output/" --checkpoint best.ckpt

### 5. Inference on new cells

`state tx infer` applies a trained run to an AnnData that isn't in the training TOML. The data should be in the same space as what you trained on, e.g. Tx1 embeddings.

In [ ]:
RUN_DIR="/path/to/state/output"
state tx infer \
  --model-dir "$RUN_DIR" \
  --checkpoint "$RUN_DIR/checkpoints/final.ckpt" \
  --adata "/path/to/input_adata.h5ad" \
  --pert-col "drugname_drugconc" \
  --embed-key "tx1-70m" \
  --output "$RUN_DIR/predictions.h5ad"